In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import os
warnings.filterwarnings("ignore")

# 1. 加载数据
print("正在加载数据...")
# 修改为与其他模型相同的数据路径
df_original = pd.read_excel('../../data/pinghua_data_x_names.xlsx')
print(f"原始数据集大小: {df_original.shape}")

# 2. 定义目标变量 - 与其他模型保持一致
target_vars = ['x25', 'x26', 'x27', 'x28', 'x29']

# 3. 分离特征和目标
X_original = df_original.drop(target_vars, axis=1)
y_original = df_original[target_vars]

# 4. 分割原始数据为训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(
    X_original, y_original, test_size=0.2, random_state=42
)

print(f"训练集大小: {X_train.shape}")
print(f"测试集大小: {X_test.shape}")

# 5. 数据标准化
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
y_train_scaled = scaler_y.fit_transform(y_train)
X_test_scaled = scaler_X.transform(X_test)

# 打印数据范围，确保标准化正常
print("\n数据标准化检查:")
print(f"X_train_scaled 均值: {np.mean(X_train_scaled):.4f}, 标准差: {np.std(X_train_scaled):.4f}")
print(f"X_train_scaled 最小值: {np.min(X_train_scaled):.4f}, 最大值: {np.max(X_train_scaled):.4f}")
print(f"y_train_scaled 均值: {np.mean(y_train_scaled):.4f}, 标准差: {np.std(y_train_scaled):.4f}")

# 6. 准备PyTorch数据集和数据加载器
class TabularDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        if y is not None:
            self.y = torch.tensor(y, dtype=torch.float32)
        else:
            self.y = None
            
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        else:
            return self.X[idx]

# 7. 定义Tab-Transformer模型
class TabTransformer(nn.Module):
    def __init__(self, input_dim, output_dim, embedding_dim=32, depth=4, 
                 heads=8, attn_dropout=0.1, ff_dropout=0.1):
        super().__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        
        # 嵌入层
        self.embedding = nn.Linear(input_dim, embedding_dim)
        
        # 位置编码
        self.pos_encoder = nn.Parameter(torch.zeros(1, input_dim, embedding_dim))
        
        # Transformer编码器层
        encoder_layers = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=heads,
            dim_feedforward=embedding_dim * 4,
            dropout=attn_dropout,
            activation='gelu',
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=depth)
        
        # MLP头部
        self.mlp_head = nn.Sequential(
            nn.Linear(embedding_dim * input_dim, embedding_dim * 2),
            nn.LayerNorm(embedding_dim * 2),
            nn.GELU(),
            nn.Dropout(ff_dropout),
            nn.Linear(embedding_dim * 2, embedding_dim),
            nn.LayerNorm(embedding_dim),
            nn.GELU(),
            nn.Dropout(ff_dropout),
            nn.Linear(embedding_dim, output_dim)
        )
        
    def forward(self, x):
        batch_size = x.size(0)
        
        # 重塑输入
        x = x.unsqueeze(-1)
        
        # 为每个特征创建嵌入
        x = x.repeat(1, 1, self.embedding.in_features)
        x = x * torch.eye(self.embedding.in_features, device=x.device)
        
        # 应用线性嵌入
        x = self.embedding(x)  # shape: [batch_size, num_features, embedding_dim]
        
        # 添加位置编码
        x = x + self.pos_encoder
        
        # 通过Transformer编码器
        x = self.transformer_encoder(x)
        
        # 重塑输出
        x = x.reshape(batch_size, -1)
        
        # 通过MLP头部
        x = self.mlp_head(x)
        
        return x

# 创建训练集和验证集
X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train_scaled, y_train_scaled, test_size=0.2, random_state=42
)

# 创建数据集
train_dataset = TabularDataset(X_train_split, y_train_split)
val_dataset = TabularDataset(X_val, y_val)
test_dataset = TabularDataset(X_test_scaled)

# 根据最优参数设置批量大小
batch_size = 65

# 创建数据加载器
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

# 训练模型
print("\n开始训练Tab-Transformer模型...")

# 创建模型
input_dim = X_train_scaled.shape[1]
output_dim = len(target_vars)

print(f"\n创建Tab-Transformer模型 (输入维度: {input_dim}, 输出维度: {output_dim})")

# 使用最优参数创建模型
model = TabTransformer(
    input_dim=input_dim,
    output_dim=output_dim,
    embedding_dim=72,     # 最优参数 8 × 9 = 72
    depth=3,              # 最优参数
    heads=1,              # 最优参数
    attn_dropout=0.3864463966779327,  # 使用精确值
    ff_dropout=0.0        # 最优参数
)

# 检查可用设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")
model = model.to(device)

# 优化器和损失函数 - 使用最优学习率
optimizer = torch.optim.Adam(model.parameters(), lr=0.0006774079054453337)  # 使用最优参数中的lr值
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-6
)
loss_fn = nn.MSELoss()

# 训练函数
def train_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    epoch_loss = 0
    
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        
        # 前向传播
        optimizer.zero_grad()
        y_pred = model(x)
        loss = loss_fn(y_pred, y)
        
        # 反向传播
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    return epoch_loss / len(loader)

# 验证函数
def validate(model, loader, loss_fn, device):
    model.eval()
    val_loss = 0
    
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            y_pred = model(x)
            loss = loss_fn(y_pred, y)
            val_loss += loss.item()
    
    return val_loss / len(loader)

# 训练模型 - 根据最优参数训练完整的100个epoch，不使用早停
print("\n开始训练Tab-Transformer模型...")
n_epochs = 100  # 使用最优参数中指定的epochs值
best_val_loss = float('inf')
best_model_state = None

train_losses = []
val_losses = []

for epoch in range(n_epochs):
    # 训练
    train_loss = train_epoch(model, train_loader, optimizer, loss_fn, device)
    train_losses.append(train_loss)
    
    # 验证
    val_loss = validate(model, val_loader, loss_fn, device)
    val_losses.append(val_loss)
    
    # 更新学习率
    scheduler.step(val_loss)
    
    # 保存最佳模型状态（不使用早停，但仍然保存最佳模型）
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = model.state_dict().copy()
    
    # 打印进度（每10个epoch）
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{n_epochs}, Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}")

print("模型训练完成！")

# 如果找到了最佳模型状态，则加载它
if best_model_state:
    model.load_state_dict(best_model_state)
    print("加载了表现最佳的模型状态")

# 在测试集上进行预测
model.eval()
all_preds = []

with torch.no_grad():
    for batch in test_loader:
        if isinstance(batch, list):
            batch = batch[0]  # 当没有标签时，DataLoader会返回一个列表
        batch = batch.to(device)
        preds = model(batch).cpu().numpy()
        all_preds.append(preds)

# 合并批次预测
y_pred_scaled = np.vstack(all_preds)

# 反标准化预测结果
y_pred = scaler_y.inverse_transform(y_pred_scaled)

# 评估模型性能
print("\n模型评估指标:")
overall_r2 = 0
overall_rmse = 0
overall_mae = 0

for idx, name in enumerate(target_vars):
    # 计算指标
    mse = mean_squared_error(y_test.iloc[:, idx], y_pred[:, idx])
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test.iloc[:, idx], y_pred[:, idx])
    r2 = r2_score(y_test.iloc[:, idx], y_pred[:, idx])
    
    # 累加指标值
    overall_r2 += r2
    overall_rmse += rmse
    overall_mae += mae
    
    # 打印每个目标变量的指标
    print(f"\n{name} 指标:")
    print(f"MSE={mse:.4f}, RMSE={rmse:.4f}, MAE={mae:.4f}, R²={r2:.4f}")

# 计算平均指标
overall_r2 /= len(target_vars)
overall_rmse /= len(target_vars)
overall_mae /= len(target_vars)

print("\n总体平均指标:")
print(f"平均 RMSE: {overall_rmse:.4f}")
print(f"平均 MAE: {overall_mae:.4f}")
print(f"平均 R²: {overall_r2:.4f}")

# 可视化训练过程
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(range(1, len(train_losses) + 1), train_losses, label='训练损失')
plt.plot(range(1, len(val_losses) + 1), val_losses, label='验证损失')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('训练和验证损失')
plt.legend()
plt.grid(True)
plt.show()